# 6. DESCARGA E INTERPRETACIÓN DE IMAGENES LANDSAT 8 Y SENTINEL 1 #

- Descarga de imagenes Landsat 8 con índices
- Inferencia de plataforma S1A / S1B
- Selección escenas:
    - Misms fecha
    - Ventana post-incendio 

<span style="color:red"> Se debe buscar alguna mascara para pluma de humo derivado de los incendios. Podría ser la causante de gran parte de la imagen no cuente con datos </span>

## 6.1  MOSAICOS MENSUALES DE LANDSAT 8 PARA EL PERIODO DE ANALISIS (2014 - 2024) ##

Genera y exportar composites mensuales de Landsat 8 Collection 2 Tier 1 Level-2 entre 2014 y 2024 para un AOI definido por shapefile. En términos operativos, el notebook: Lee un AOI desde shapefile y lo transforma a ee.Geometry en WGS84 (EPSG:4326), aplicando correcciones de geometría (validación, buffer(0), unión). Construye una colección mensual (ImageCollection) filtrada por fecha y AOI.

Aplica: enmascaramiento QA (nubes/sombras y saturación radiométrica), escalado a reflectancia para bandas SR, 
una máscara topográfica vía hillShadow (control de sombra por relieve), y calcula índices espectrales NDVI, EVI, NBR y NDWI. Calcula un composite mediana por mes. Añade una banda IMG_COUNT con el número de imágenes del mes.

Exporta cada composite a Google Drive como GeoTIFF (30 m), creando 11×12 = 132 tareas.

###  6.1.1 Parte 1 ###

Autenticación e inicialización de GEE; lectura y preparación del área de interes, reproyectar y validar geometria. Definición de funciones e índices. Se realiza mascara topografica usando el SRTM y geometría solar, devolviendo imagenes con bandas corregidas. Se emplean el índice NDVI, EVI, NBR y NDWI. Se define las fechas para el mosaico, filtrandose por fecha y área de interes. Se obtiene la mediana del mes y el número de iamgenes empleadas para el mosaico. Finalmente se exporta a google drive con resolcuión de 30 metros y se genera un reporte final por tareas. Los datos aquí suministrados, de momento, se manejan es sistema de coordenadas EPSG = 4326. 


In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
import ee
from src.gee_aoi import load_aoi_ee_geometry
from src.gee_export import setup_logger, run_monthly_exports

# 1) Auth solo cuando toque
ee.Authenticate()
ee.Initialize()

ruta_aoi_shp = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19.shp"
aoi = load_aoi_ee_geometry(ruta_aoi_shp, target_epsg=4326)

# params
year_start, year_end = 2014, 2024
folder_drive = "Imagenes_GEE"
scale = 30
dry_run = False  # True para probar sin lanzar tareas

log_path = os.path.join(os.getcwd(), "logs", "gee_exports.log")
logger = setup_logger(log_path)

tasks = run_monthly_exports(
    aoi=aoi,
    year_start=year_start,
    year_end=year_end,
    folder=folder_drive,
    scale=scale,
    logger=logger,
    dry_run=dry_run
)

print("Total tareas:", len(tasks))
tasks[:5]

Total tareas: 132


[TaskInfo(year=2014, month=1, desc='L8_2014_01'),
 TaskInfo(year=2014, month=2, desc='L8_2014_02'),
 TaskInfo(year=2014, month=3, desc='L8_2014_03'),
 TaskInfo(year=2014, month=4, desc='L8_2014_04'),
 TaskInfo(year=2014, month=5, desc='L8_2014_05')]

## 6.2 Preprocesamiento Imagenes Sentinel 1 GRD ##

Consulta la colección Sentinel-1 GRD (COPERNICUS/S1_GRD) para un año específico (YEAR_TO_RUN) y un sentido orbital (ASCENDING o DESCENDING). Genera dos productos complementarios: Un CSV índice (metadatos por escena) exportado a Google Drive; exportación de imágenes procesadas por lotes, con preprocesamiento SAR (máscara de borde, normalización gamma0, ratio VV/VH, filtrado speckle multitemporal y diferencia VV respecto a una referencia). Se descargaron 458 imagenes en orbita ascendente entre diciembre de 2014 y diciembre de 2021. No se encontraron imagenes para meses anteriores a diciembre de 2014 e incosistencia de imagenes mensuales para 2015 y 2016. No se encontraron imagenes de orbita ascendente desde el año 2022 al 2023.

<font color="red">
    
<font color="red"> Dudas :</font>

El filtrado de speckle basado en imágenes previas podría sesgar la caracterización del área quemada. Aunque en el modelo Random Forest se seleccionen imágenes de un mes posterior al evento (por ejemplo, del 04 de febrero para un incendio ocurrido el 04 de enero), si el proceso de reducción de ruido utiliza un histórico anterior a esa fecha, la  de retrodispersión del incendio podría suavizarse o alterarse con información del estado previo de la vegetación.

Todo esta en dB por lo que VV_Difference y VV_ratio no darían bien

Se deberia calcular en lineal, conserva el signo físico, y se pasa a “dB con signo” con sign(diff_lin)*10*log10(max(|diff_lin|, epsilon)) no sé si sería correcto de esta forma
</font>

### 6.2.1 Parte 1 ###

Inicializa GEE; construye el área de interes como ee.Geometry desde shapefile y define parámetros globales (año, órbita, tamaño de lote); define funciones de preprocesamiento SAR con orbita ascendente. Construcción de la colección Sentinel 1, definiendose la selección de escenas Sentienl 1 que intersectan el área de interes, estan dentro del año, son IW GRD, cuentan con polarizaciones VV y VH y para la orbita previamente definida (ascendente). Se realiza Speckle multitemporal basado en imágenes previas. Para cada imagen, se construye una mediana de las N_PREV_IMAGES inmediatamente anteriores (en tiempo), y luego suavizar espacialmente con un kernel boxcar. Se calcula Banda de diferencia VV respecto a referencia, permitiendo realizar cambios respecto a una condición base del año. Se exporta CSV con los metadatos de las imagenes

<font color="red">
    
El siguiente codigo no genera vv/vh ratio ni vv_difference aún. El speackel deberia realizar primero, antes de la normalización 
</font>

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
import logging
from pathlib import Path
import sys
import ee

import ee
from src.pipeline_sentinel1 import build_collection
from src.radiometry import border_noise_mask, gamma0_db, terrain_flattening
from src.speckle import _db_to_lin, _lin_to_db, refined_lee_spatial
from src.indices import _db_to_lin, _lin_to_db, add_ratio_db, add_vv_difference_1y_signed_db

In [4]:


# =========================
# 0) PROJECT ROOT (una sola verdad)
# =========================
PROJECT_DIR = Path.cwd().resolve()

# Validaciones rápidas (si algo falla aquí, el problema es de ruta real)
print("PROJECT_DIR:", PROJECT_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())

CFG_PATH = PROJECT_DIR / "configs" / "s1_default.json"
LOG_PATH = PROJECT_DIR / "logs" / "run.log"

print("PROJECT_DIR:", PROJECT_DIR)
print("CFG_PATH:", CFG_PATH)
print("CFG exists:", CFG_PATH.exists())

# Si esto imprime False, el archivo NO está ahí (o la letra D: no es la que crees)
# y por eso "sigue el mismo error".
if not CFG_PATH.exists():
    raise FileNotFoundError(f"No existe el config en: {CFG_PATH}")

# =========================
# 1) Logging reproducible (en ruta ABSOLUTA)
# =========================
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    filename=str(LOG_PATH),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logging.info("=== START RUN ===")

# =========================
# 2) Asegurar imports de src/
# =========================
# Necesitas que PROJECT_DIR esté en sys.path para 'import src...'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# =========================
# 3) Earth Engine init
# =========================
ee.Authenticate()
ee.Initialize()

# =========================
# 4) Cargar config
# =========================
cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))
print("Config cargado OK")

# =========================
# 5) AOI desde shapefile
# =========================
from src.aoi import aoi_shp_to_ee_geometry

AOI_PATH = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19.shp"
aoi = aoi_shp_to_ee_geometry(AOI_PATH)

# =========================
# 6) Pipeline (IMPORTA build_pipeline)
# =========================
from src.pipeline_sentinel1 import build_pipeline  # <-- AJUSTA si tu módulo se llama distinto

col_final = build_pipeline(aoi, cfg)

# =========================
# 7) Validación (getInfo SOLO aquí)
# =========================
bands = ee.Image(col_final.first()).bandNames().getInfo()
n = col_final.size().getInfo()

print("Bandas:", bands)
print("N imágenes:", n)

logging.info(f"bands={bands}")
logging.info(f"n_images={n}")

PROJECT_DIR: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20
PROJECT_DIR exists: True
PROJECT_DIR: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20
CFG_PATH: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\configs\s1_default.json
CFG exists: True


Enter verification code:  4/1AfrIepDAiXo7I947nD4e46vcH-h59bptjugqgUwovogU5MGy4j682PLUNWQ



Successfully saved authorization token.
Config cargado OK
Bandas: ['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference']
N imágenes: 92


In [8]:
import ee
from src.pipeline_sentinel1 import build_collection
from src.radiometry import border_noise_mask, gamma0_db, terrain_flattening
from src.speckle import _db_to_lin, _lin_to_db, refined_lee_spatial
from src.indices import _db_to_lin, _lin_to_db, add_ratio_db, add_vv_difference_1y_signed_db



def show_bands(col, label):
    img = ee.Image(col.first())
    print(label, "->", img.bandNames().getInfo())

def show_size(col, label):
    print(label, "size ->", col.size().getInfo())

col0 = build_collection(aoi, int(cfg["year"]), str(cfg["orbit"]))
show_size(col0, "0) build_collection")
show_bands(col0, "0) build_collection")

col1 = col0.map(border_noise_mask)
show_bands(col1, "1) +border_noise_mask")

col2 = col1.map(gamma0_db)
show_bands(col2, "2) +gamma0_db")

dem = ee.Image(str(cfg["dem_id"]))
col3 = col2.map(lambda img: terrain_flattening(img, dem=dem))
show_bands(col3, "3) +terrain_flattening")

col4 = col3.map(lambda img: refined_lee_spatial(img, kernel_size=int(cfg["kernel_size"])))
show_bands(col4, "4) +refined_lee_spatial")

col5 = col4.map(lambda img: add_ratio_db(img, eps=float(cfg["eps"])))
show_bands(col5, "5) +add_ratio_db")

col6 = add_vv_difference_1y_signed_db(col5, eps=float(cfg["eps"]))
show_bands(col6, "6) +add_vv_difference_1y_signed_db")
show_size(col6, "FINAL")

0) build_collection size -> 92
0) build_collection -> ['VV', 'VH', 'angle']
1) +border_noise_mask -> ['VV', 'VH', 'angle']
2) +gamma0_db -> ['VV', 'VH', 'angle']
3) +terrain_flattening -> ['VV', 'VH', 'angle']
4) +refined_lee_spatial -> ['VV', 'VH', 'angle']
5) +add_ratio_db -> ['VV', 'VH', 'angle', 'VVVH_ratio']
6) +add_vv_difference_1y_signed_db -> ['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference']
FINAL size -> 92


In [9]:
EXPORT_BANDS = ["VV", "VH", "angle", "VVVH_ratio", "VV_Difference"]

def export_batch_to_drive(col, aoi, year, folder, scale, max_pixels,
                          batch_start=0, batch_size=10):
    # 1) tamaño (getInfo SOLO aquí)
    n = col.size().getInfo()
    print(f"[EXPORT] year={year} -> n_images={n}")
    if n == 0:
        print("[EXPORT] Colección vacía. No se crean tareas.")
        return []

    batch_end = min(batch_start + batch_size, n)
    if batch_start >= n:
        raise ValueError(f"batch_start={batch_start} >= n={n}")

    img_list = col.toList(n)

    tasks = []
    for i in range(batch_start, batch_end):
        img = ee.Image(img_list.get(i))

        # VALIDACIÓN ligera (opcional): asegura bandas antes de exportar
        # (solo para depurar 1 imagen; comenta luego)
        # print("bands_i:", img.bandNames().getInfo())

        out = img.select(EXPORT_BANDS).toFloat()

        desc = f"S1_{year}_{i}"
        task = ee.batch.Export.image.toDrive(
            image=out,
            description=desc,
            folder=folder,
            fileNamePrefix=desc,
            region=aoi,
            scale=scale,
            maxPixels=max_pixels,
            fileFormat="GeoTIFF"
        )
        task.start()
        tasks.append(task)

    print(f"[EXPORT] Tareas lanzadas: {len(tasks)} ({batch_start}..{batch_end-1})")
    print("[EXPORT] Siguiente batch_start sugerido:", batch_end)
    return tasks

In [10]:
col_final = build_pipeline(aoi, cfg)

# 1) Verifica bandas en servidor (una vez)
print(ee.Image(col_final.first()).bandNames().getInfo())

# 2) Exporta
tasks = export_batch_to_drive(
    col_final,
    aoi=aoi,
    year=int(cfg["year"]),
    folder="Imagenes_GEE",
    scale=30,
    max_pixels=1e13,
    batch_start=25,
    batch_size=5
)

['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference']
[EXPORT] year=2019 -> n_images=92
[EXPORT] Tareas lanzadas: 5 (25..29)
[EXPORT] Siguiente batch_start sugerido: 30


### 6.2.2  Parte 2 ###

Selección de mejores escenas Sentinel 1 por mes. Debido a que se contaba con varias imagenes por mes, se busca identificar la imagen que mayor representatividad tenga del mes. Se implementa un flujo de control de calidad y selección objetiva de escenas Sentinel-1 a partir de un conjunto de GeoTIFF (patrón S1_201910*.tif). El objetivo operativo es: Recortar cada raster al área de interes y homogenizar NoData como NaN; medir validez de píxeles por banda y global; calcular estadística descriptiva por banda y por escena; generar histogramas para inspección visual; ejecutar un ranking final para seleccionar la mejor escena, priorizando: máximo porcentaje de píxeles válidos en todas las bandas, mínimo porcentaje de nulos en cualquier banda, desempate por coherencia geométrica y opcionalmente y coherencia orbital/slice usando un CSV índice.

Se identifica que la mayoria de imagenes son similares para los meses de analisis por lo que pdría escogerse cualquiera

Definir rutas, parámetros y configuración de salida; leer AOI, limpiar geometrías y validar. Inferir orden de bandas y loop por cada TIFF: máscara con el área de interes, inferencia de bandas, huella geométrica y comparación de grilla, métricas de validez por banda y global, estadística descriptiva por banda e histograma por banda. Se contruye reporte y ranking para establecer el orden, desempate por grilla y desempate por orbita. Selección de la mejor escena y reporte tecnico.

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from pathlib import Path
import logging

from src.config_histogra_sentinel1 import load_config
from src.pipeline_histograma_sentinel1 import run_qc_batch

PROJECT = Path.cwd()
cfg = load_config(PROJECT / "configs" / "qc_s1_batch.json")

log_path = PROJECT / "logs" / "qc_s1_batch.log"
log_path.parent.mkdir(exist_ok=True)

logging.basicConfig(
    filename=str(log_path),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("qc_s1_batch")

res = run_qc_batch(cfg, logger)

display(res["df_consolidated_rank"].head(20))
print("Reporte consolidado:", res["paths"]["global_report_txt"])
print("Ranking consolidado:", res["paths"]["global_rank_csv"])

,folder,file,path,total_px_aoi,bands_used,valid_allbands_px,valid_allbands_pct,null_anyband_px,null_anyband_pct,geom_match_ref,...,VH_null_px,VH_null_pct,angle_valid_px,angle_valid_pct,angle_null_px,angle_null_pct,VVVH_ratio_valid_px,VVVH_ratio_valid_pct,VVVH_ratio_null_px,VVVH_ratio_null_pct
0,S1_2015_09,S1_2015_4.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4837824,"VV,VH,angle,VVVH_ratio",1327162,27.433036,3510662,72.566964,True,...,3482878,71.992656,1449662,29.965166,3388162,70.034834,1354946,28.007344,3482878,71.992656
1,S1_2015_11,S1_2015_5.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",1325275,27.404318,3510733,72.595682,True,...,3482974,72.021676,1451505,30.014529,3384503,69.985471,1353034,27.978324,3482974,72.021676
2,S1_2015_05,S1_2015_2.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",1214224,25.107982,3621784,74.892018,False,...,3596859,74.376614,1491285,30.837108,3344723,69.162892,1239149,25.623386,3596859,74.376614
3,S1_2016_11,S1_2016_16.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",824733,17.054004,4011275,82.945996,False,...,4011275,82.945996,1456265,30.112957,3379743,69.887043,1355731,28.034093,3480277,71.965907
4,S1_2016_09,S1_2016_7.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4837824,"VV,VH,angle,VVVH_ratio",823585,17.023873,4014239,82.976127,False,...,3483963,72.015084,1453501,30.044520,3384323,69.955480,1353861,27.984916,3483963,72.015084
5,S1_2016_09,S1_2016_6.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",822716,17.012296,4013292,82.987704,True,...,3483813,72.039025,1423172,29.428653,3412836,70.571347,1352195,27.960975,3483813,72.039025
6,S1_2016_12,S1_2016_19.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",821662,16.990501,4014346,83.009499,True,...,4014346,83.009499,1451340,30.011117,3384668,69.988883,1350292,27.921625,3485716,72.078375
7,S1_2016_11,S1_2016_13.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",821268,16.982354,4014740,83.017646,True,...,3486279,72.090017,1450161,29.986737,3385847,70.013263,1349729,27.909983,3486279,72.090017
8,S1_2017_05,S1_2017_25.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4837824,"VV,VH,angle,VVVH_ratio",821358,16.977840,4016466,83.022160,False,...,3487360,72.085301,1450283,29.978003,3387541,70.021997,1350464,27.914699,3487360,72.085301
9,S1_2016_03,S1_2016_2.tif,D:\Maestria_Geomatica\Semestre_IV\Tesis_Python...,4836008,"VV,VH,angle,VVVH_ratio",819978,16.955679,4016030,83.044321,True,...,3488400,72.133876,1547738,32.004455,3288270,67.995545,819978,16.955679,4016030,83.044321


Reporte consolidado: outputs\qc_s1_batch_noVVDIFF\reporte_consolidado.txt
Ranking consolidado: outputs\qc_s1_batch_noVVDIFF\ranking_consolidado.csv
